# 15 · Voronoi catchments against shapely

**Question.** Does Fieldwork's `voronoi` widget partition the ground the way GEOS does?

This is the first step of the John Snow example and the one step of its chain that had no
check. With it, the whole example — cells, clip, count — is independently recomputed.

**Two unrelated algorithms.** Fieldwork starts from a rectangle and clips it with one
half-plane per other site: the set of points at least as close to this site as to that one, in
projected metres. The reference asks GEOS for the Voronoi diagram of the same projected points
and clips it to the same envelope. Neither method informs the other, which is the strongest
form this lab's comparisons take.

The envelope is **recomputed here from the boundary** by the declared rule — the projected
corners of the reporting area's bounding box, their extreme easting and northing, expanded by
100 m — rather than taken from the fixture, so that derivation is tested too.

Criteria fixed before running: each cell within 1 m² of GEOS's region for the same site, the
cells tiling the envelope with no overlap, and every cell containing its own site.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_15_voronoi import run

report = run(Path.cwd() / "fixtures" / "voronoi-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["sites"], "sites")
print(f"worst symmetric difference   {report['worstSymmetricDifferenceSquareMetres']:.3e} m2")
print(f"union against the envelope   {report['unionAgainstEnvelopeSquareMetres']:.3e} m2")
print(f"pairwise overlap             {report['pairwiseOverlapSquareMetres']:.3e} m2")
print(f"every cell holds its site    {report['everyCellHoldsItsSite']}")
print()
print(f"{'site':<12} {'cell area m2':>14} {'symmetric difference':>22}")
for row in report["perCell"]:
    print(f"{row['siteId']:<12} {row['areaSquareMetres']:14.0f} {row['symmetricDifferenceSquareMetres']:22.3e}")

## Dive deeper · the assumption is straight-line distance

A Voronoi cell is the ground nearer this pump than any other **by straight-line distance in
a projection**. Nobody walks in straight lines, and in 1854 Soho they walked on streets.

Fieldwork says this in the widget's note — "cells model proximity, not actual pump use" — and
the project's own answer to it is a different widget: `reachable_corridor`, which walks the
street network. The two are directly comparable, and the comparison is the lesson.

In [ ]:
# How much of the straight-line cell is actually within a 5 minute walk?
# The corridor fixture holds the walked answer for the Broad Street pump.
import numpy as np, pyproj, shapely
from shapely.geometry import shape

voronoi_fixture = json.loads((Path.cwd() / "fixtures" / "voronoi-soho-001.json").read_text())
corridor_fixture = json.loads((Path.cwd() / "fixtures" / "corridor-soho-001.json").read_text())
to_metric = pyproj.Transformer.from_crs("OGC:CRS84", "EPSG:32630", always_xy=True)
project = lambda geometry: shapely.transform(
    geometry, lambda coords: np.column_stack(to_metric.transform(coords[:, 0], coords[:, 1])))

broad_street = next(c for c in voronoi_fixture["output"]["cells"] if c["siteId"] == "pump-250")
cell = project(shape(broad_street["geometry"]))
walked = project(shape(corridor_fixture["output"]["corridor"]))

print(f"nearest-pump cell, straight line   {cell.area:10.0f} m2")
print(f"5 minute walk along the streets    {walked.area:10.0f} m2")
print(f"walkable ground inside the cell    {cell.intersection(walked).area:10.0f} m2")
print(f"walkable ground outside the cell   {walked.difference(cell).area:10.0f} m2")
print()
print("The two answer different questions. A cell says which pump is nearest as the crow")
print("flies; a corridor says where you can actually get to in five minutes. Neither is a")
print("statement about which pump anybody used.")

### Reading the result

`agrees` means two unrelated algorithms partitioned the same ground to within a few millionths
of a square metre. It does not make the partition meaningful: Euclidean distance in a
projection ignores streets, walls and the Thames, and the reference ignores them identically.